In [1]:
import os
import sys
import json
import glob
import math
import shutil
import numpy as np
import pandas as pd
import cv2
from scipy import signal
from scipy.signal import periodogram
from tqdm import tqdm
import torch
import torch.multiprocessing
torch.multiprocessing.set_sharing_strategy('file_system')
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

REPO_ROOT = "/home/naver/disk2/HoangDPB/rPPG-Toolbox"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from neural_methods.model.DeepPhys import DeepPhys
from neural_methods.model.TS_CAN import TSCAN

In [2]:
# ----- paths -----
RAW_DATA_PATH       = os.path.join(REPO_ROOT, "raw_data/reading")
PREPROCESSED_PATH   = os.path.join(REPO_ROOT, "preprocessed_data/groupA")
OUTPUT_DIR          = os.path.join(REPO_ROOT, "results/groupA/reading")

# ----- video / signal params -----
VIDEO_FPS    = 30       # camera frame rate
PPG_FS       = 25       # PPG sensor sampling rate (Hz)

# ----- Group A preprocessing params -----
CHUNK_LENGTH = 180      # frames per clip (6 s at 30 fps)
IMG_H, IMG_W = 72, 72   # face crop resolution
LABEL_TYPE   = "DiffNormalized"  # needs cumsum in post-processing
DATA_FORMAT  = "NDCHW"  # (N, D, C, H, W)
NUM_CHANNELS = 6        # DiffNormalized (3) + Standardized (3)

# ----- device -----
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

os.makedirs(PREPROCESSED_PATH, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("PREPROCESSED_PATH:", PREPROCESSED_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

Device: cuda:0
PREPROCESSED_PATH: /home/naver/disk2/HoangDPB/rPPG-Toolbox/preprocessed_data/groupA
OUTPUT_DIR: /home/naver/disk2/HoangDPB/rPPG-Toolbox/results/groupA/reading


In [3]:
# Model selection toggle
# Uncomment additional models as needed; each entry is (display_name, model_class_key, weight_path).
# model_class_key: "DeepPhys" or "Tscan"

MODELS = [
    ("PURE_DeepPhys",              "DeepPhys", "final_model_release/PURE_DeepPhys.pth"),
    ("PURE_TSCAN",                 "Tscan",    "final_model_release/PURE_TSCAN.pth"),
    ("SCAMPS_DeepPhys",          "DeepPhys", "final_model_release/SCAMPS_DeepPhys.pth"),
    ("SCAMPS_TSCAN",             "Tscan",    "final_model_release/SCAMPS_TSCAN.pth"),
    ("UBFC-rPPG_DeepPhys",       "DeepPhys", "final_model_release/UBFC-rPPG_DeepPhys.pth"),
    ("UBFC-rPPG_TSCAN",          "Tscan",    "final_model_release/UBFC-rPPG_TSCAN.pth"),
    ("BP4D_PseudoLabel_DeepPhys","DeepPhys", "final_model_release/BP4D_PseudoLabel_DeepPhys.pth"),
    ("BP4D_PseudoLabel_TSCAN",   "Tscan",    "final_model_release/BP4D_PseudoLabel_TSCAN.pth"),
    ("MA-UBFC_deepphys",         "DeepPhys", "final_model_release/MA-UBFC_deepphys.pth"),
    ("MA-UBFC_tscan",            "Tscan",    "final_model_release/MA-UBFC_tscan.pth"),
]

# TS-CAN frame_depth (TSM window size) — must match training config
TSCAN_FRAME_DEPTH = 10

print(f"Selected {len(MODELS)} model(s):")
for name, cls, path in MODELS:
    print(f"  {name}  ({cls})  ->  {path}")

Selected 10 model(s):
  PURE_DeepPhys  (DeepPhys)  ->  final_model_release/PURE_DeepPhys.pth
  PURE_TSCAN  (Tscan)  ->  final_model_release/PURE_TSCAN.pth
  SCAMPS_DeepPhys  (DeepPhys)  ->  final_model_release/SCAMPS_DeepPhys.pth
  SCAMPS_TSCAN  (Tscan)  ->  final_model_release/SCAMPS_TSCAN.pth
  UBFC-rPPG_DeepPhys  (DeepPhys)  ->  final_model_release/UBFC-rPPG_DeepPhys.pth
  UBFC-rPPG_TSCAN  (Tscan)  ->  final_model_release/UBFC-rPPG_TSCAN.pth
  BP4D_PseudoLabel_DeepPhys  (DeepPhys)  ->  final_model_release/BP4D_PseudoLabel_DeepPhys.pth
  BP4D_PseudoLabel_TSCAN  (Tscan)  ->  final_model_release/BP4D_PseudoLabel_TSCAN.pth
  MA-UBFC_deepphys  (DeepPhys)  ->  final_model_release/MA-UBFC_deepphys.pth
  MA-UBFC_tscan  (Tscan)  ->  final_model_release/MA-UBFC_tscan.pth


In [4]:
# Read video frames

def read_video_frames(video_path):
    """Read all frames from an MP4 file.

    Returns:
        frames (np.ndarray): shape (T, H, W, 3), dtype uint8, RGB order.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")

    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()

    if not frames:
        raise ValueError(f"Empty video: {video_path}")
    return np.stack(frames, axis=0)

In [5]:
# Read and time-align PPG signal

def read_ppg_synced(session_path, num_frames, fps=30):
    """Read PPG green channel from ppg.csv and resample to video frame times.

    The PPG sensor timestamps and video start timestamp are both in ms
    Unix epoch stored in metadata.json (sync_markers.video_start).
    Resampling is done by linear interpolation onto frame-spaced timestamps.

    Returns:
        ppg (np.ndarray): shape (num_frames,), dtype float32, raw green values.
    """
    meta_path = os.path.join(session_path, "metadata.json")
    with open(meta_path, "r") as f:
        meta = json.load(f)
    video_start_ms = meta["sync_markers"].get("video_start")
    if video_start_ms is None:
        video_start_ms = meta.get("start_timestamp")
    if video_start_ms is None:
        raise ValueError(f"No video_start or start_timestamp found in {meta_path}")
    video_start_ms = float(video_start_ms)

    ppg_df = pd.read_csv(os.path.join(session_path, "ppg.csv"))
    ppg_ts    = ppg_df["timestamp"].values.astype(np.float64)
    ppg_green = ppg_df["green"].values.astype(np.float64)

    # convert ppg timestamps to seconds relative to video start
    ppg_t = (ppg_ts - video_start_ms) / 1000.0

    # frame timestamps in seconds
    frame_t = np.arange(num_frames, dtype=np.float64) / fps

    # clip frame times to valid ppg range to avoid extrapolation
    frame_t_clipped = np.clip(frame_t, ppg_t[0], ppg_t[-1])

    ppg_resampled = np.interp(frame_t_clipped, ppg_t, ppg_green)
    return ppg_resampled.astype(np.float32)

In [6]:
# Normalization functions

def diff_normalize_data(data):
    """DiffNormalized: (frame[t+1]-frame[t]) / (frame[t+1]+frame[t]+1e-7), then / std."""
    data = data.astype(np.float32)
    n, h, w, c = data.shape
    out = np.zeros_like(data)
    out[:n - 1] = (data[1:] - data[:-1]) / (data[1:] + data[:-1] + 1e-7)
    std = np.std(out)
    if std > 0:
        out /= std
    return out


def standardized_data(data):
    """Standardized: global z-score over all pixels and frames."""
    data = data.astype(np.float32)
    m = np.mean(data)
    s = np.std(data)
    if s > 0:
        data = (data - m) / s
    else:
        data = np.zeros_like(data)
    data = np.where(np.isnan(data), np.zeros_like(data), data)
    return data


def diff_normalize_label(label):
    """DiffNormalized label: finite difference normalised by std, zero-padded."""
    diff = np.diff(label.astype(np.float64), axis=0)
    s = np.std(diff)
    if s > 0:
        diff = diff / s
    return np.append(diff, [0.0]).astype(np.float32)

In [7]:
# Face crop + resize

def crop_face_resize(frames, out_h, out_w, large_box_coef=1.5):
    """Detect face on frame 0 with Haar Cascade, expand bbox by coef, resize all frames."""
    xml_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    detector  = cv2.CascadeClassifier(xml_path)

    frame0 = frames[0]
    if frame0.dtype != np.uint8:
        frame0 = np.clip(frame0, 0, 255).astype(np.uint8)
    gray = cv2.cvtColor(frame0, cv2.COLOR_RGB2GRAY)

    faces = detector.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)
    H, W = frames.shape[1], frames.shape[2]

    if len(faces) > 0:
        x, y, fw, fh = max(faces, key=lambda f: f[2])  # largest face
        x  = max(0, int(x  - (large_box_coef - 1.0) / 2.0 * fw))
        y  = max(0, int(y  - (large_box_coef - 1.0) / 2.0 * fh))
        fw = min(int(fw * large_box_coef), W - x)
        fh = min(int(fh * large_box_coef), H - y)
    else:
        x, y, fw, fh = 0, 0, W, H  # fallback: full frame

    C = frames.shape[3]
    resized = np.zeros((len(frames), out_h, out_w, C), dtype=np.float32)
    for i, frame in enumerate(frames):
        crop = frame[y : y + fh, x : x + fw]
        if crop.size == 0:
            crop = frame
        resized[i] = cv2.resize(crop.astype(np.float32), (out_w, out_h),
                                interpolation=cv2.INTER_AREA)
    return resized

In [8]:
# Discover subjects + read device-measured HR from hr.csv

def read_gt_hr(session_path):
    """Return average HR from hr.csv (hrStatus==1, hr>0).

    hr.csv has IBI columns whose values are arrays like [720, 695],
    which contain commas and break the standard CSV parser.
    Reading only the first 5 columns avoids the issue.
    """
    hr_path = os.path.join(session_path, "hr.csv")
    df = pd.read_csv(
        hr_path,
        usecols=[0, 1, 2, 3, 4],
        names=["id", "dataReceived", "timestamp", "hr", "hrStatus"],
        header=0,
    )
    valid = df[(df["hrStatus"] == 1) & (df["hr"] > 0)]["hr"]
    if len(valid) == 0:
        return float("nan")
    return float(valid.mean())


# Discover subject folders directly -- no metadata.csv required.
all_dirs = sorted([
    d for d in glob.glob(os.path.join(RAW_DATA_PATH, "*"))
    if os.path.isdir(d) and os.path.basename(d) != "videos"
])
print(f"Found {len(all_dirs)} subject folders\n")

subjects = []

for subj_dir in all_dirs:
    subj_id  = os.path.basename(subj_dir)
    subj_key = subj_id.replace("_", "")

    session_dirs = glob.glob(os.path.join(RAW_DATA_PATH, subj_id, "*"))
    if not session_dirs:
        print(f"No session folder for {subj_id}, skipping.")
        continue
    session_path = session_dirs[0]

    session_name  = os.path.basename(session_path)
    video_pattern = os.path.join(RAW_DATA_PATH, "videos",
                                  f"{subj_id}_{session_name}.mp4")
    video_files = glob.glob(video_pattern)
    if not video_files:
        print(f"No video found for {subj_id}, skipping.")
        continue
    video_path = video_files[0]

    gt_hr = read_gt_hr(session_path)

    subjects.append({
        "subj_id":      subj_id,
        "subj_key":     subj_key,
        "video_path":   video_path,
        "session_path": session_path,
        "gt_hr":        gt_hr,
    })
    print(f"  {subj_id}  HR_device={gt_hr:.4f} bpm  video={os.path.basename(video_path)}")

print(f"\nTotal subjects: {len(subjects)}")

Found 5 subject folders

  S_000  HR_device=80.4930 bpm  video=S_000_reading_1772099952592.mp4
  S_001  HR_device=87.9710 bpm  video=S_001_reading_1772100748921.mp4
  S_002  HR_device=74.0128 bpm  video=S_002_reading_1772101439887.mp4
  S_003  HR_device=65.4512 bpm  video=S_003_reading_1772102619267.mp4
  S_004  HR_device=76.8256 bpm  video=S_004_reading_1772103172990.mp4

Total subjects: 5


In [9]:
# Data preprocessing
# Face crop -> DiffNormalized (3ch) + Standardized (3ch) = 6ch
# Chunk into clips of CHUNK_LENGTH frames, save as .npy

# Clear any previous preprocessed data
if os.path.exists(PREPROCESSED_PATH):
    shutil.rmtree(PREPROCESSED_PATH)
os.makedirs(PREPROCESSED_PATH)
print(f"Cleared and recreated: {PREPROCESSED_PATH}\n")

all_input_files = []

for subj in subjects:
    subj_key     = subj["subj_key"]
    video_path   = subj["video_path"]
    session_path = subj["session_path"]

    print(f"=== Processing {subj_key} ===")

    # Read video
    frames = read_video_frames(video_path)
    T = frames.shape[0]
    print(f"  Video: {T} frames @ {VIDEO_FPS} fps")

    # Read and resample PPG
    ppg_signal = read_ppg_synced(session_path, T, fps=VIDEO_FPS)
    print(f"  PPG green: min={ppg_signal.min():.0f}, max={ppg_signal.max():.0f}")

    # Face crop + resize to 72x72
    frames_cropped = crop_face_resize(frames, IMG_H, IMG_W)

    # Compute DiffNormalized (3ch) and Standardized (3ch)
    diff_data = diff_normalize_data(frames_cropped)   # (T, H, W, 3)
    std_data  = standardized_data(frames_cropped)     # (T, H, W, 3)

    # Concatenate along channel axis -> 6 channels
    data_6ch = np.concatenate([diff_data, std_data], axis=-1)  # (T, H, W, 6)

    # DiffNormalized label
    label = diff_normalize_label(ppg_signal)  # (T,)

    # Chunk into clips
    clip_num = T // CHUNK_LENGTH
    data_clips  = np.array([data_6ch[i*CHUNK_LENGTH:(i+1)*CHUNK_LENGTH] for i in range(clip_num)])
    label_clips = np.array([label[i*CHUNK_LENGTH:(i+1)*CHUNK_LENGTH]    for i in range(clip_num)])

    # Save per-subject
    subj_dir = os.path.join(PREPROCESSED_PATH, subj_key)
    os.makedirs(subj_dir)

    subj_files = []
    for chunk_idx in range(clip_num):
        input_path = os.path.join(subj_dir, f"{subj_key}_input{chunk_idx}.npy")
        label_path = os.path.join(subj_dir, f"{subj_key}_label{chunk_idx}.npy")

        np.save(input_path, data_clips[chunk_idx])   # (CHUNK_LENGTH, H, W, 6)
        np.save(label_path, label_clips[chunk_idx])  # (CHUNK_LENGTH,)
        subj_files.append(input_path)

    all_input_files.extend(subj_files)
    print(f"  {clip_num} clips -> {subj_dir}\n")

print(f"Total clips saved: {len(all_input_files)}")
print("\nFolder structure:")
for subj in subjects:
    d = os.path.join(PREPROCESSED_PATH, subj["subj_key"])
    n = len(glob.glob(os.path.join(d, "*_input*.npy")))
    print(f"  {subj['subj_key']}/  ({n} clips)")

Cleared and recreated: /home/naver/disk2/HoangDPB/rPPG-Toolbox/preprocessed_data/groupA

=== Processing S000 ===
  Video: 2626 frames @ 30 fps
  PPG green: min=-71545, max=194479
  14 clips -> /home/naver/disk2/HoangDPB/rPPG-Toolbox/preprocessed_data/groupA/S000

=== Processing S001 ===
  Video: 2619 frames @ 30 fps
  PPG green: min=-227743, max=258441
  14 clips -> /home/naver/disk2/HoangDPB/rPPG-Toolbox/preprocessed_data/groupA/S001

=== Processing S002 ===
  Video: 2695 frames @ 30 fps
  PPG green: min=-187828, max=197372
  14 clips -> /home/naver/disk2/HoangDPB/rPPG-Toolbox/preprocessed_data/groupA/S002

=== Processing S003 ===
  Video: 2750 frames @ 30 fps
  PPG green: min=-401001, max=258429
  15 clips -> /home/naver/disk2/HoangDPB/rPPG-Toolbox/preprocessed_data/groupA/S003

=== Processing S004 ===
  Video: 2835 frames @ 30 fps
  PPG green: min=-17061, max=226073
  15 clips -> /home/naver/disk2/HoangDPB/rPPG-Toolbox/preprocessed_data/groupA/S004

Total clips saved: 72

Folder str

In [10]:
# PyTorch Dataset  (NDCHW format, returns (D, 6, H, W) per clip)

class GroupADataset(Dataset):

    def __init__(self, input_files):
        self.inputs = sorted(input_files)
        self.labels = [
            f.replace("input", "label")
            for f in self.inputs
        ]

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, index):
        data  = np.float32(np.load(self.inputs[index]))   # (D, H, W, 6)
        label = np.float32(np.load(self.labels[index]))   # (D,)

        # NDHWC -> NDCHW: (D, H, W, 6) -> (D, 6, H, W)
        data = np.transpose(data, (0, 3, 1, 2))

        fname      = os.path.basename(self.inputs[index])
        split_idx  = fname.index("_")
        subject_id = fname[:split_idx]                     # e.g. "S000"
        chunk_id   = fname[split_idx + 6:].split(".")[0]  # +6 skips "_input"

        return data, label, subject_id, chunk_id


dataset = GroupADataset(all_input_files)
loader  = DataLoader(dataset, batch_size=4, shuffle=False, num_workers=4)

print(f"Dataset : {len(dataset)} clips")
print(f"Loader  : {len(loader)} batches (batch_size=4)")

Dataset : 72 clips
Loader  : 18 batches (batch_size=4)


In [11]:
# Post-processing functions

def detrend(signal_in, lambda_val=100):
    """Smoothness-priors detrending (Tarvainen et al.)."""
    T_len = len(signal_in)
    H_mat = np.eye(T_len)
    ones  = np.ones(T_len)
    D_mat = (np.diag(ones[:-2], -2)
             - 2 * np.diag(ones[:-1], -1)
             + np.diag(ones))
    D_mat = D_mat[2:, :]
    inv   = np.linalg.inv(H_mat + lambda_val ** 2 * D_mat.T @ D_mat)
    return (H_mat - inv) @ signal_in


def bandpass_filter(sig, fs, low, high, order=1):
    """Zero-phase Butterworth bandpass filter."""
    b, a = signal.butter(order, [low / fs * 2, high / fs * 2], btype="bandpass")
    return signal.filtfilt(b, a, sig.astype(np.float64))


def fft_peak_hz(sig, fs, low, high):
    """Return dominant frequency (Hz) in [low, high] Hz via FFT."""
    N = 1
    while N < len(sig):
        N *= 2
    freqs, pxx = periodogram(sig, fs=fs, nfft=N, detrend=False)
    mask = (freqs >= low) & (freqs <= high)
    if not mask.any():
        return 0.0
    return float(freqs[mask][np.argmax(pxx[mask])])


def calculate_snr(pred_ppg, hr_label_bpm, fs, low_pass=0.6, high_pass=3.3):
    """Signal-to-noise ratio at HR harmonics vs background noise (dB)."""
    N = 1
    while N < len(pred_ppg):
        N *= 2
    freqs, pxx = periodogram(pred_ppg, fs=fs, nfft=N, detrend=False)

    f1  = hr_label_bpm / 60.0
    f2  = 2 * f1
    dev = 6.0 / 60.0  # +-6 bpm tolerance

    sig_mask   = (((freqs >= f1 - dev) & (freqs <= f1 + dev))
                  | ((freqs >= f2 - dev) & (freqs <= f2 + dev)))
    noise_mask = ((freqs >= low_pass) & (freqs <= high_pass) & ~sig_mask)

    sig_power   = pxx[sig_mask].sum()
    noise_power = pxx[noise_mask].sum()
    if noise_power == 0:
        return float("inf")
    return float(10.0 * np.log10(sig_power / noise_power))


def _reform_from_dict(chunk_dict):
    """Concatenate chunks in sorted key order into a 1-D array."""
    return np.concatenate([chunk_dict[k] for k in sorted(chunk_dict.keys())])


def process_bvp(pred_chunks, label_chunks, fs=30, diff_flag=True):
    """Post-process predicted and label BVP chunk dicts into HR estimates.

    When diff_flag=True (DiffNormalized labels), signals are cumsum'd
    before detrending to recover the original BVP waveform.
    """
    pred  = _reform_from_dict(pred_chunks).astype(np.float64)
    label = _reform_from_dict(label_chunks).astype(np.float64)

    if diff_flag:
        pred  = detrend(np.cumsum(pred),  100)
        label = detrend(np.cumsum(label), 100)
    else:
        pred  = detrend(pred,  100)
        label = detrend(label, 100)

    pred_processed  = bandpass_filter(pred,  fs, low=0.6, high=3.3)
    label_processed = bandpass_filter(label, fs, low=0.6, high=3.3)

    hr_pred  = fft_peak_hz(pred_processed,  fs, 0.6, 3.3) * 60.0
    hr_label = fft_peak_hz(label_processed, fs, 0.6, 3.3) * 60.0
    snr_db   = calculate_snr(pred_processed, hr_label, fs)

    return hr_pred, hr_label, snr_db, pred_processed

In [12]:
# Blink rate detection

def detect_blink_rate(video_path, fps=30, large_box_coef=1.5):
    """Estimate blink rate (blinks/min) using eye-strip brightness analysis.

    Steps:
      1. Detect face on frame 0 with Haar Cascade.
      2. Define eye strip = rows [20%, 50%] of face bbox height.
      3. Compute mean grayscale brightness in eye strip per frame.
      4. Invert (blinks = dips) and bandpass filter [0.1, 0.9] Hz.
      5. Count peaks with minimum separation of 1 second.
    """
    frames = read_video_frames(video_path)
    T = len(frames)

    xml_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    detector = cv2.CascadeClassifier(xml_path)
    gray0 = cv2.cvtColor(frames[0], cv2.COLOR_RGB2GRAY)
    faces = detector.detectMultiScale(gray0, scaleFactor=1.3, minNeighbors=5)
    H, W = frames[0].shape[:2]

    if len(faces) > 0:
        x, y, fw, fh = max(faces, key=lambda f: f[2])
        x  = max(0, int(x  - (large_box_coef - 1.0) / 2.0 * fw))
        y  = max(0, int(y  - (large_box_coef - 1.0) / 2.0 * fh))
        fw = min(int(fw * large_box_coef), W - x)
        fh = min(int(fh * large_box_coef), H - y)
    else:
        x, y, fw, fh = 0, 0, W, H

    eye_top    = y + int(fh * 0.20)
    eye_bottom = y + int(fh * 0.50)
    eye_left   = x
    eye_right  = x + fw

    brightness = np.array([
        np.mean(cv2.cvtColor(f, cv2.COLOR_RGB2GRAY)[eye_top:eye_bottom, eye_left:eye_right])
        for f in frames
    ])

    # blinks = dips in brightness -> invert for find_peaks
    brightness_inv = -brightness.astype(np.float64)

    # bandpass [0.1, 0.9] Hz = [6, 54] blinks/min
    b, a = signal.butter(2, [0.1 / fps * 2, 0.9 / fps * 2], btype="bandpass")
    filtered = signal.filtfilt(b, a, brightness_inv)

    # minimum 1 second between detected blinks
    min_dist = int(fps)
    peaks, _ = signal.find_peaks(filtered, distance=min_dist)

    duration_min = T / fps / 60.0
    return len(peaks) / duration_min


# Compute blink rate for each subject
blink_rate_dict = {}
for subj in subjects:
    subj_key   = subj["subj_key"]
    video_path = subj["video_path"]
    print(f"  {subj_key} ...", end=" ", flush=True)
    blink_rate = detect_blink_rate(video_path, fps=VIDEO_FPS)
    blink_rate_dict[subj_key] = blink_rate
    print(f"{blink_rate:.2f} blinks/min")

print("\nBlink rates:", {k: f"{v:.2f}" for k, v in blink_rate_dict.items()})

  S000 ... 30.85 blinks/min
  S001 ... 30.24 blinks/min
  S002 ... 29.39 blinks/min
  S003 ... 28.80 blinks/min
  S004 ... 33.65 blinks/min

Blink rates: {'S000': '30.85', 'S001': '30.24', 'S002': '29.39', 'S003': '28.80', 'S004': '33.65'}


In [13]:
# Main inference loop
# For each model in MODELS: load -> infer -> per-subject results -> aggregate metrics
# -> export metrics.json + ppg_results.csv into results_groupA/{model_name}/

FS = VIDEO_FPS

# lookup: subj_key -> subject info dict
subjects_by_key = {s["subj_key"]: s for s in subjects}

# blink_rate_dict is populated by the blink detection cell;
# fall back to empty dict if that cell has not been run yet.
if "blink_rate_dict" not in dir():
    blink_rate_dict = {}

for model_display_name, model_class_key, model_rel_path in MODELS:
    model_path = os.path.join(REPO_ROOT, model_rel_path)
    print(f"\n{'='*70}")
    print(f"Model: {model_display_name}  ({model_class_key})")
    print(f"Weights: {model_path}")
    print(f"{'='*70}\n")

    # ---- Instantiate model ----
    if model_class_key == "DeepPhys":
        model = DeepPhys(img_size=IMG_H)
        frame_depth = None  # no TSM alignment needed
    elif model_class_key == "Tscan":
        model = TSCAN(frame_depth=TSCAN_FRAME_DEPTH, img_size=IMG_H)
        frame_depth = TSCAN_FRAME_DEPTH
    else:
        raise ValueError(f"Unknown model_class_key: {model_class_key}")

    # ---- Load weights ----
    state_dict = torch.load(model_path, map_location=DEVICE)

    # strip 'module.' prefix if present (DataParallel artifact)
    if any(k.startswith("module.") for k in state_dict.keys()):
        state_dict = {k[len("module."):]: v for k, v in state_dict.items()}

    model.load_state_dict(state_dict)
    model = model.to(DEVICE)
    model.eval()

    num_params = sum(p.numel() for p in model.parameters())
    print(f"Model loaded. Total parameters: {num_params:,}")

    # ---- Inference ----
    bvp_preds_dict  = {}  # subj_key -> {chunk_id: np.ndarray (CHUNK_LENGTH,)}
    bvp_labels_dict = {}

    with torch.no_grad():
        for batch in tqdm(loader, desc=f"Inference [{model_display_name}]"):
            data_t, labels_t, batch_subjects, batch_chunk_ids = batch

            N, D, C, H, W = data_t.shape  # (N, CHUNK_LENGTH, 6, 72, 72)

            # Flatten batch + temporal -> (N*D, 6, H, W)
            data_flat   = data_t.view(N * D, C, H, W).to(DEVICE)
            labels_flat = labels_t.view(N * D)  # (N*D,)

            # For TS-CAN: trim to multiple of frame_depth (TSM alignment)
            if frame_depth is not None:
                trim = (N * D) // frame_depth * frame_depth
            else:
                trim = N * D  # DeepPhys: no alignment needed

            data_flat   = data_flat[:trim]
            labels_flat = labels_flat[:trim]

            pred = model(data_flat)            # (trim, 1)
            pred_np  = pred.squeeze(-1).cpu().numpy()      # (trim,)
            label_np = labels_flat[:trim].numpy()           # (trim,)

            for i in range(N):
                start = i * CHUNK_LENGTH
                end   = start + CHUNK_LENGTH
                if end > trim:
                    break

                subj = batch_subjects[i]
                cid  = int(batch_chunk_ids[i])

                if subj not in bvp_preds_dict:
                    bvp_preds_dict[subj]  = {}
                    bvp_labels_dict[subj] = {}

                bvp_preds_dict[subj][cid]  = pred_np[start:end]
                bvp_labels_dict[subj][cid] = label_np[start:end]

    print(f"\nInference complete. Subjects: {sorted(bvp_preds_dict.keys())}")

    # ---- Per-subject results ----
    per_subject_results = []
    hr_preds_all = []
    gt_hrs_all   = []
    snr_all      = []

    print(f"\n{'Subject':<10} {'HR_pred':>10} {'HR_gt':>10} {'HR_err':>8} {'SNR':>7} {'Blinks':>7}")
    print("-" * 60)

    for subj_key in sorted(bvp_preds_dict.keys()):
        hr_pred, hr_label, _, pred_processed = process_bvp(
            bvp_preds_dict[subj_key], bvp_labels_dict[subj_key],
            fs=FS, diff_flag=True
        )

        gt_hr      = subjects_by_key[subj_key]["gt_hr"]
        snr_db     = calculate_snr(pred_processed, gt_hr, FS)
        blink_rate = blink_rate_dict.get(subj_key, float("nan"))
        hr_err     = hr_pred - gt_hr

        # map subj_key ("S000") back to original ID ("S_000")
        subj_id = subj_key[0] + "_" + subj_key[1:]

        per_subject_results.append({
            "name":                subj_id,
            "predicted_heartrate": hr_pred,
            "real_heartrate":      gt_hr,
            "heartrate_error":     hr_err,
            "blink_rate":          blink_rate,
            "snr_db":              snr_db,
        })

        hr_preds_all.append(hr_pred)
        gt_hrs_all.append(gt_hr)
        snr_all.append(snr_db)

        print(f"{subj_id:<10} {hr_pred:>10.3f} {gt_hr:>10.3f} {hr_err:>8.3f} {snr_db:>7.2f} {blink_rate:>7.2f}")

    hr_preds_all = np.array(hr_preds_all)
    gt_hrs_all   = np.array(gt_hrs_all)
    snr_all      = np.array(snr_all)

    # ---- Aggregate metrics ----
    n = len(hr_preds_all)
    assert n > 0, "No subjects to evaluate."

    err   = hr_preds_all - gt_hrs_all
    abs_e = np.abs(err)
    sq_e  = err ** 2
    rel_e = abs_e / (np.abs(gt_hrs_all) + 1e-9)

    mae       = float(np.mean(abs_e))
    mae_se    = float(np.std(abs_e) / np.sqrt(n))

    rmse      = float(np.sqrt(np.mean(sq_e)))
    rmse_se   = float(np.sqrt(np.std(sq_e) / np.sqrt(n)))

    mape      = float(np.mean(rel_e) * 100.0)
    mape_se   = float(np.std(rel_e) / np.sqrt(n) * 100.0)

    if n >= 2:
        pearson_r  = float(np.corrcoef(hr_preds_all, gt_hrs_all)[0, 1])
        pearson_se = float(np.sqrt(max(0.0, (1 - pearson_r ** 2) / (n - 2))))
    else:
        pearson_r, pearson_se = float("nan"), float("nan")

    mean_snr    = float(np.mean(snr_all))
    mean_snr_se = float(np.std(snr_all) / np.sqrt(n))

    print(f"\n--- Aggregate Metrics [{model_display_name}] ---")
    print(f"MAE     : {mae:.4f} +/- {mae_se:.4f} bpm")
    print(f"RMSE    : {rmse:.4f} +/- {rmse_se:.4f} bpm")
    print(f"MAPE    : {mape:.4f} +/- {mape_se:.4f} %")
    print(f"Pearson : {pearson_r:.4f} +/- {pearson_se:.4f}")
    print(f"SNR     : {mean_snr:.4f} +/- {mean_snr_se:.4f} dB")

    # ---- Export to per-model subdirectory ----
    model_output_dir = os.path.join(OUTPUT_DIR, model_display_name)
    os.makedirs(model_output_dir, exist_ok=True)

    metrics_dict = {
        "model":      model_display_name,
        "n_subjects": n,
        "evaluation_method": "FFT",
        "bvp_bandpass_hz":   [0.6, 3.3],
        "aggregate_metrics": {
            "MAE":     {"value": mae,       "se": mae_se,      "unit": "bpm"},
            "RMSE":    {"value": rmse,      "se": rmse_se,     "unit": "bpm"},
            "MAPE":    {"value": mape,      "se": mape_se,     "unit": "%"},
            "Pearson": {"value": pearson_r, "se": pearson_se,  "unit": ""},
            "SNR":     {"value": mean_snr,  "se": mean_snr_se, "unit": "dB"},
        },
        "per_subject": [
            {
                "name":               r["name"],
                "predicted_heartrate": r["predicted_heartrate"],
                "real_heartrate":      r["real_heartrate"],
                "heartrate_error":     r["heartrate_error"],
                "snr_db":              r["snr_db"],
            }
            for r in per_subject_results
        ],
    }

    json_path = os.path.join(model_output_dir, "metrics.json")
    with open(json_path, "w") as fh:
        json.dump(metrics_dict, fh, indent=2)
    print(f"\nMetrics saved to: {json_path}")

    # ---- Export ppg_results.csv ----
    csv_rows = []
    for r in per_subject_results:
        csv_rows.append({
            "name":               r["name"],
            "predicted_heartrate": r["predicted_heartrate"],
            "real_heartrate":      r["real_heartrate"],
            "heartrate_error":     r["heartrate_error"],
            "blink_rate":          r["blink_rate"],
        })

    results_df = pd.DataFrame(csv_rows, columns=[
        "name", "predicted_heartrate", "real_heartrate",
        "heartrate_error", "blink_rate"
    ])

    csv_path = os.path.join(model_output_dir, "ppg_results.csv")
    results_df.to_csv(csv_path, index=False)

    print(f"CSV saved to: {csv_path}")
    print()
    print(results_df.to_string(index=False))

print(f"\n\nAll models processed. Results in: {OUTPUT_DIR}")


Model: PURE_DeepPhys  (DeepPhys)
Weights: /home/naver/disk2/HoangDPB/rPPG-Toolbox/final_model_release/PURE_DeepPhys.pth

Model loaded. Total parameters: 2,228,643


Inference [PURE_DeepPhys]: 100%|██████████| 18/18 [00:02<00:00,  8.16it/s]



Inference complete. Subjects: ['S000', 'S001', 'S002', 'S003', 'S004']

Subject       HR_pred      HR_gt   HR_err     SNR  Blinks
------------------------------------------------------------
S_000          70.752     80.493   -9.741   -5.78   30.85
S_001          62.842     87.971  -25.129   -6.20   30.24
S_002          73.389     74.013   -0.624   -1.35   29.39
S_003          49.658     65.451  -15.793   -3.61   28.80
S_004          76.904     76.826    0.079   -4.20   33.65

--- Aggregate Metrics [PURE_DeepPhys] ---
MAE     : 10.2732 +/- 4.2355 bpm
RMSE    : 13.9727 +/- 10.2818 bpm
MAPE    : 13.1484 +/- 5.2191 %
Pearson : 0.4084 +/- 0.5270
SNR     : -4.2267 +/- 0.7737 dB

Metrics saved to: /home/naver/disk2/HoangDPB/rPPG-Toolbox/results/groupA/reading/PURE_DeepPhys/metrics.json
CSV saved to: /home/naver/disk2/HoangDPB/rPPG-Toolbox/results/groupA/reading/PURE_DeepPhys/ppg_results.csv

  name  predicted_heartrate  real_heartrate  heartrate_error  blink_rate
 S_000            70.751953

Inference [PURE_TSCAN]: 100%|██████████| 18/18 [00:01<00:00,  9.34it/s]



Inference complete. Subjects: ['S000', 'S001', 'S002', 'S003', 'S004']

Subject       HR_pred      HR_gt   HR_err     SNR  Blinks
------------------------------------------------------------
S_000          77.344     80.493   -3.149   -4.62   30.85
S_001          89.209     87.971    1.238   -5.68   30.24
S_002          73.389     74.013   -0.624   -0.98   29.39
S_003          64.600     65.451   -0.852   -4.83   28.80
S_004          76.904     76.826    0.079   -4.05   33.65

--- Aggregate Metrics [PURE_TSCAN] ---
MAE     : 1.1883 +/- 0.4695 bpm
RMSE    : 1.5856 +/- 1.2925 bpm
MAPE    : 1.5133 +/- 0.5745 %
Pearson : 0.9845 +/- 0.1013
SNR     : -4.0301 +/- 0.7205 dB

Metrics saved to: /home/naver/disk2/HoangDPB/rPPG-Toolbox/results/groupA/reading/PURE_TSCAN/metrics.json
CSV saved to: /home/naver/disk2/HoangDPB/rPPG-Toolbox/results/groupA/reading/PURE_TSCAN/ppg_results.csv

  name  predicted_heartrate  real_heartrate  heartrate_error  blink_rate
 S_000            77.343750       80.492

...


Inference complete. Subjects: ['S000', 'S001', 'S002', 'S003', 'S004']

Subject       HR_pred      HR_gt   HR_err     SNR  Blinks
------------------------------------------------------------
S_000          65.039     80.493  -15.454   -7.23   30.85
S_001          58.008     87.971  -29.963   -5.16   30.24
S_002          70.752     74.013   -3.261   -0.63   29.39
S_003          57.568     65.451   -7.883   -6.08   28.80
S_004          76.465     76.826   -0.361   -4.27   33.65

--- Aggregate Metrics [BP4D_PseudoLabel_TSCAN] ---
MAE     : 11.3843 +/- 4.7392 bpm
RMSE    : 15.5533 +/- 12.3120 bpm
MAPE    : 14.0357 +/- 5.3231 %
Pearson : -0.0361 +/- 0.5770
SNR     : -4.6740 +/- 1.0063 dB

Metrics saved to: /home/naver/disk2/HoangDPB/rPPG-Toolbox/results/groupA/reading/BP4D_PseudoLabel_TSCAN/metrics.json
CSV saved to: /home/naver/disk2/HoangDPB/rPPG-Toolbox/results/groupA/reading/BP4D_PseudoLabel_TSCAN/ppg_results.csv

  name  predicted_heartrate  real_heartrate  heartrate_error  blink_rate

Inference [MA-UBFC_deepphys]: 100%|██████████| 18/18 [00:01<00:00,  9.69it/s]



Inference complete. Subjects: ['S000', 'S001', 'S002', 'S003', 'S004']

Subject       HR_pred      HR_gt   HR_err     SNR  Blinks
------------------------------------------------------------
S_000          80.859     80.493    0.366   -3.19   30.85
S_001          87.012     87.971   -0.959   -2.32   30.24
S_002          70.752     74.013   -3.261    0.53   29.39
S_003          64.600     65.451   -0.852    0.89   28.80
S_004          76.465     76.826   -0.361    0.11   33.65

--- Aggregate Metrics [MA-UBFC_deepphys] ---
MAE     : 1.1598 +/- 0.4824 bpm
RMSE    : 1.5839 +/- 1.3499 bpm
MAPE    : 1.5444 +/- 0.6571 %
Pearson : 0.9883 +/- 0.0880
SNR     : -0.7959 +/- 0.7346 dB

Metrics saved to: /home/naver/disk2/HoangDPB/rPPG-Toolbox/results/groupA/reading/MA-UBFC_deepphys/metrics.json
CSV saved to: /home/naver/disk2/HoangDPB/rPPG-Toolbox/results/groupA/reading/MA-UBFC_deepphys/ppg_results.csv

  name  predicted_heartrate  real_heartrate  heartrate_error  blink_rate
 S_000            80.8

Inference [MA-UBFC_tscan]: 100%|██████████| 18/18 [00:01<00:00,  9.39it/s]



Inference complete. Subjects: ['S000', 'S001', 'S002', 'S003', 'S004']

Subject       HR_pred      HR_gt   HR_err     SNR  Blinks
------------------------------------------------------------
S_000          79.541     80.493   -0.952   -2.34   30.85
S_001          88.330     87.971    0.359   -2.53   30.24
S_002          72.949     74.013   -1.064    0.01   29.39
S_003          65.039     65.451   -0.412    1.32   28.80
S_004          76.904     76.826    0.079    1.04   33.65

--- Aggregate Metrics [MA-UBFC_tscan] ---
MAE     : 0.5731 +/- 0.1674 bpm
RMSE    : 0.6845 +/- 0.4526 bpm
MAPE    : 0.7520 +/- 0.2200 %
Pearson : 0.9978 +/- 0.0383
SNR     : -0.5001 +/- 0.7330 dB

Metrics saved to: /home/naver/disk2/HoangDPB/rPPG-Toolbox/results/groupA/reading/MA-UBFC_tscan/metrics.json
CSV saved to: /home/naver/disk2/HoangDPB/rPPG-Toolbox/results/groupA/reading/MA-UBFC_tscan/ppg_results.csv

  name  predicted_heartrate  real_heartrate  heartrate_error  blink_rate
 S_000            79.541016    